In [0]:
data = [
    ("Tahnik","Ahmed",'Raiyan',"M",2305),
    ("Esrak","Fardin","Ratul","M",9100),
    ("Rasel","Shikder","Rifat","F",2400),
    ("Sajib","Ahmed","Shobuj","M",14000),
    ("Sahria","Hossain","Sajjad","M",71000)

]
column = ["First_Name","Middle_Name","Last_Name","Gender","Salary"]
df1 = spark.createDataFrame(data = data,schema= column)


In [0]:
df1.show()

+----------+-----------+---------+------+------+
|First_Name|Middle_Name|Last_Name|Gender|Salary|
+----------+-----------+---------+------+------+
|    Tahnik|      Ahmed|   Raiyan|     M|  2305|
|     Esrak|     Fardin|    Ratul|     M|  9100|
|     Rasel|    Shikder|    Rifat|     F|  2400|
|     Sajib|      Ahmed|   Shobuj|     M| 14000|
|    Sahria|    Hossain|   Sajjad|     M| 71000|
+----------+-----------+---------+------+------+



In [0]:
import pyspark
from pyspark.sql import SparkSession 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType



read data from csv and dbfs location 


In [0]:
df = spark.read.load("/FileStore/tables/googleplaystore-3.csv",format= 'csv',sep= ',',header = 'true',inferschema = True,escape = '"' )


In [0]:
df.count()

Out[46]: 10841

In [0]:
df.show(2)

+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+-----------+------------+
|                 App|      Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|              Genres|    Last Updated|Current Ver| Android Ver|
+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+-----------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159| 19M| 10,000+|Free|    0|      Everyone|        Art & Design| January 7, 2018|      1.0.0|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967| 14M|500,000+|Free|    0|      Everyone|Art & Design;Pret...|January 15, 2018|      2.0.0|4.0.3 and up|
+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+-----------+------------+
only showing top 2 rows



In [0]:
df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



Data Celaning 

In [0]:
df = df.drop('Last Updated','Size','Content Rating','Current Ver')

In [0]:
df.show(2)

+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
|                 App|      Category|Rating|Reviews|Installs|Type|Price|              Genres| Android Ver|
+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159| 10,000+|Free|    0|        Art & Design|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967|500,000+|Free|    0|Art & Design;Pret...|4.0.3 and up|
+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
only showing top 2 rows



Changing the datatypes of the columns 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import IntegerType
df = df.withColumn("Reviews",col("Reviews").cast(IntegerType()))\
  .withColumn("Installs",regexp_replace(col("Installs"), "[^0-9]", ""))\
      .withColumn("Installs",col("Installs").cast(IntegerType()))\
          .withColumn("Price",col("Price").cast(IntegerType()))
  


In [0]:
df.show(2)

+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
|                 App|      Category|Rating|Reviews|Installs|Type|Price|              Genres| Android Ver|
+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159|   10000|Free|    0|        Art & Design|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967|  500000|Free|    0|Art & Design;Pret...|4.0.3 and up|
+--------------------+--------------+------+-------+--------+----+-----+--------------------+------------+
only showing top 2 rows



implementing sql on the pyspark dataframe 

In [0]:
df.createOrReplaceTempView('app')

In [0]:
%sql select * from app

App,Category,Rating,Reviews,Installs,Type,Price,Genres,Android Ver
Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,10000,Free,0,Art & Design,4.0.3 and up
Coloring book moana,ART_AND_DESIGN,3.9,967,500000,Free,0,Art & Design;Pretend Play,4.0.3 and up
"U Launcher Lite – FREE Live Cool Themes, Hide Apps",ART_AND_DESIGN,4.7,87510,5000000,Free,0,Art & Design,4.0.3 and up
Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,50000000,Free,0,Art & Design,4.2 and up
Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,100000,Free,0,Art & Design;Creativity,4.4 and up
Paper flowers instructions,ART_AND_DESIGN,4.4,167,50000,Free,0,Art & Design,2.3 and up
Smoke Effect Photo Maker - Smoke Editor,ART_AND_DESIGN,3.8,178,50000,Free,0,Art & Design,4.0.3 and up
Infinite Painter,ART_AND_DESIGN,4.1,36815,1000000,Free,0,Art & Design,4.2 and up
Garden Coloring Book,ART_AND_DESIGN,4.4,13791,1000000,Free,0,Art & Design,3.0 and up
Kids Paint Free - Drawing Fun,ART_AND_DESIGN,4.7,121,10000,Free,0,Art & Design;Creativity,4.0.3 and up


Finding the app with top reviews 

In [0]:
%sql select * from app
order by Reviews desc 
limit 10;

App,Category,Rating,Reviews,Installs,Type,Price,Genres,Android Ver
Facebook,SOCIAL,4.1,78158306,1000000000,Free,0,Social,Varies with device
Facebook,SOCIAL,4.1,78128208,1000000000,Free,0,Social,Varies with device
WhatsApp Messenger,COMMUNICATION,4.4,69119316,1000000000,Free,0,Communication,Varies with device
WhatsApp Messenger,COMMUNICATION,4.4,69119316,1000000000,Free,0,Communication,Varies with device
WhatsApp Messenger,COMMUNICATION,4.4,69109672,1000000000,Free,0,Communication,Varies with device
Instagram,SOCIAL,4.5,66577446,1000000000,Free,0,Social,Varies with device
Instagram,SOCIAL,4.5,66577313,1000000000,Free,0,Social,Varies with device
Instagram,SOCIAL,4.5,66577313,1000000000,Free,0,Social,Varies with device
Instagram,SOCIAL,4.5,66509917,1000000000,Free,0,Social,Varies with device
Messenger – Text and Video Chat for Free,COMMUNICATION,4.0,56646578,1000000000,Free,0,Communication,Varies with device


In [0]:
%sql select App , sum(Reviews) as TotalReview, avg(Rating) as Average_Rating
from app
group by app
order by TotalReview Desc
limit 10;

App,TotalReview,Average_Rating
Instagram,266241989,4.5
WhatsApp Messenger,207348304,4.4
Clash of Clans,179558781,4.6
Messenger – Text and Video Chat for Free,169932272,4.0
Subway Surfers,166331958,4.5
Candy Crush Saga,156993136,4.3999999999999995
Facebook,156286514,4.1
8 Ball Pool,99386198,4.5
Clash Royale,92530298,4.6
Snapchat,68045010,4.0


Top Installed Apps

In [0]:
%sql select App , sum (installs) as Total_Installation 
from app 
group by 1 
order by 2 Desc;

App,Total_Installation
Subway Surfers,6000000000
Instagram,4000000000
Hangouts,4000000000
Google Drive,4000000000
Google News,4000000000
Google Photos,4000000000
Candy Crush Saga,3500000000
Google Chrome: Fast & Secure,3000000000
WhatsApp Messenger,3000000000
Messenger – Text and Video Chat for Free,3000000000


Catgeory Wise Installation 

In [0]:
%sql select Category , sum(installs)
from app 
group by 1
order by 2 desc;

Category,sum(installs)
GAME,35086024415
COMMUNICATION,32647276251
PRODUCTIVITY,14176091369
SOCIAL,14069867902
TOOLS,11452771915
FAMILY,10258263505
PHOTOGRAPHY,10088247655
NEWS_AND_MAGAZINES,7496317760
TRAVEL_AND_LOCAL,6868887146
VIDEO_PLAYERS,6222002720


Databricks visualization. Run in Databricks to view.

top paid apps 

In [0]:
%sql select App , sum(Price)
from app 
where Type = 'Paid'
group by 1
order by 2 desc; 

App,sum(Price)
I am Rich Plus,null
Trine 2: Complete Story,null
Whoowasit? - Best kids game!,null
AF-STROKE,null
Sokoban Land DX,null
Servidor Privado CR y CoC - Royale Servers PRO,null
Campervan.Guide Pro,null
XCOM®: Enemy Within,null
SweetLand — Family Board Game,null
Medical ID - In Case of Emergency (ICE),null


In [0]:
%sql select distinct Type from app

Type
0
NaN
Free
Paid
